#Enterprise Fleet Analytics Pipeline: Focuses on the business outcome (analytics) and the domain (fleet/logistics).

![logistics](https://raw.githubusercontent.com/iamashok1410/databricks-code-repo/main/4_logistics_usecase/logistics_project.png)

Download the data from the below gdrive and upload into the catalog
https://drive.google.com/drive/folders/1J3AVJIPLP7CzT15yJIpSiWXshu1iLXKn?usp=drive_link

In [0]:
%sql
create catalog if not exists logistic_catalog;
create database if not exists logistic_catalog.default;
create volume if not exists logistic_catalog.default.logistic_volume; 

In [0]:
dbutils.fs.mkdirs("/Volumes/logistic_catalog/default/logistic_volume/logistics_data")

data path:
/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_source1<br>
/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_source2<br>
/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_shipment_detail_3000.json

##**1. Data Munging** -

####1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)

- It is a Structured data with comma seperator (CSV)
-  No comments, footer is there in the data
- Total columns are (seperator + 1)
- Data Quality
  - Null rows and columns are there
  - duplicate rows & Duplicate id keys
  - format issues are there (age is not in number format eg. ten)
  - Number of columns are more or less than the expected
  example: 5000007,Amit,Patel,45 And 5000006,John,Mathews,ten,Supervisor,Additionalcolumn
- Identification of data type




####2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
1. Apply inferSchema and toDF to create a DF and analyse the actual data.
2. Analyse the schema, datatypes, columns etc.,
3. Analyse the duplicate records count and summary of the dataframe.

In [0]:
rawdf1=spark.read.csv("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_source1",inferSchema=True,header=True).toDF("shipment_id","first_name","last_name","age","role")
rawdf1.show(30,False)
display(rawdf1.take(20))
print(rawdf1.columns)
print(rawdf1.dtypes)
print(rawdf1.schema)
print(rawdf1.printSchema())

In [0]:
print("actual count of the record", rawdf1.count())
print("count of depulicated record of all columns",rawdf1.dropDuplicates().count())
print("count of deplicated record of shipment id column", rawdf1.dropDuplicates(["shipment_id"]).count())
display(rawdf1.describe())
display(rawdf1.summary())

###a. Passive Data Munging -  (File: logistics_source1  and logistics_source2)
Without modifying the data, identify:<br>
Shipment IDs that appear in both master_v1 and master_v2<br>
Records where:<br>
1. shipment_id is non-numeric
2. age is not an integer<br>

Count rows having:
3. fewer columns than expected
4. more columns than expected

In [0]:
#Create a Spark Session Object
from pyspark.sql.session import SparkSession
spark1 = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

###**b. Active Data Munging** File: logistics_source1 and logistics_source2

#####1.Combining Data + Schema Merging (Structuring)
1. Read both files without enforcing schema
2. Align them into a single canonical schema: shipment_id,
first_name,
last_name,
age,
role,
hub_location,
vehicle_type,
data_source
3. Add data_source column with values as: system1, system2 in the respective dataframes

**1.Read both files without enforcing schema**

In [0]:
rawdf2 = spark.read.csv("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_source1", header=True, inferSchema=False)
print(rawdf2.printSchema())
rawdf2.show(30)

rawdf3 = spark.read.csv("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/logistics_source2", header=True, inferSchema=False)
print(rawdf3.printSchema())
rawdf3.show(30)

**2.Align them into a single canonical schema: shipment_id, first_name, last_name, age, role, hub_location, vehicle_type, data_source**<br>
**3.Add data_source column with values as: system1, system2 in the respective dataframes**

In [0]:
from pyspark.sql.functions import lit,col
rawdf2=rawdf2.withColumn("data_source",lit("system1"))
rawdf3=rawdf3.withColumn("data_source",lit("system2"))
display(rawdf2)
display(rawdf3)
rawdf_merged=rawdf2.unionByName(rawdf3,allowMissingColumns=True).select("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type","data_source")
print(rawdf_merged.printSchema())
display(rawdf_merged)


#####2. Cleansing, Scrubbing: 
Cleansing (removal of unwanted datasets)<br>
1. Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role<br>
2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name<br>
3. Join Readiness Rule - Drop records where the join key is null: shipment_id<br>

Scrubbing (convert raw to tidy)<br>
4. Age Defaulting Rule - Fill NULL values in the age column with: -1<br>
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN<br>
6. Invalid Age Replacement - Replace the following values in age:
"ten" to -1
"" to -1<br>
7. Vehicle Type Normalization - Replace inconsistent vehicle types: 
truck to LMV
bike to TwoWheeler

**Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role**

In [0]:
cleans_df=rawdf_merged.where("shipment_id is not null and role is not null")
display(cleans_df)
print(cleans_df.count())

**Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name**

In [0]:
cleans_df1=cleans_df.where("first_name is not null or last_name is not null")
display(cleans_df1)
display(cleans_df1.count())

Join Readiness Rule - Drop records where the join key is null: shipment_id

In [0]:
cleans_df2=cleans_df1.where("shipment_id is not null")
display(cleans_df2)
display(cleans_df2.count())

**Scrubbing (convert raw to tidy)
4. Age Defaulting Rule - Fill NULL values in the age column with: -1**

In [0]:
from pyspark.sql.functions import coalesce,lit
cleans_df3=cleans_df2.na.replace({"ten":"10"},["age"])
scrub_df1=cleans_df3.withColumn("age",coalesce(col("age").cast("int"),lit(-1)))
display(scrub_df1.show())

**5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN**

In [0]:
scrub_df2=scrub_df1.withColumn("vehicle_type",coalesce(col("vehicle_type"),lit("UNKNOWN")))
display(scrub_df2.show())

6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1

In [0]:
scrub_df3=scrub_df2.na.replace({10:-1},subset=["age"])
display(scrub_df3.show())

**7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler**

In [0]:
scrub_df4=scrub_df3.na.replace({"Truck":"LMV","Bike":"TwoWheeler"},subset=["vehicle_type"])
display(scrub_df4)

####3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

Detail Dataframe creation <br>
1. Read Data from logistics_shipment_detail.json
2. As this data is a clean json data, it doesn't require any cleansing or scrubbing.

**Read Data from logistics_shipment_detail.json**

In [0]:
strct2="shipment_id string,order_id string,source_city string,destination_city string,shipment_status string,cargo_type string,vehicle_type string,payment_mode string,shipment_weight_kg string,shipment_cost string,shipment_date string"
rawdf_jsn=spark.read.schema(strct2).json("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/", multiLine="True",pathGlobFilter="logistics_shipment_*")
display(rawdf_jsn)
display(rawdf_jsn.count())

Standardizations:<br>

1. Add a column<br> 
Source File: logistics_shipment_detail_3000.json<br>: domain as 'Logistics'
2. Column Uniformity: 
role - Convert to lowercase<br>
Source File: logistics_source1 & logistics_source2<br>
vehicle_type - Convert values to UPPERCASE<br>
Source Files: logistics_shipment_detail_3000.json (and the merged master files)
hub_location - Convert values to initcap case<br>
3. Format Standardization:<br>
Source Files: logistics_shipment_detail_3000.json
Convert shipment_ref to string<br>
Pad to 10 characters with leading zeros<br>
Convert dispatch_date to yyyy-MM-dd<br>
Ensure delivery_cost has 2 decimal precision<br>
4. Data Type Standardization<br>
Standardizing column data types to fix schema drift and enable mathematical operations.<br>
Source File: logistics_source1 & logistics_source2 <br>
age: Cast String to Integer<br>
Source File: logistics_shipment_detail_3000.json<br>
shipment_weight_kg: Cast to Double<br>
Source File: logistics_shipment_detail_3000.json<br>
is_expedited: Cast to Boolean<br>
5. Naming Standardization <br>
Source File: logistics_source1 & logistics_source2<br>
Rename: first_name to staff_first_name<br>
Rename: last_name to staff_last_name<br>
Rename: hub_location to origin_hub_city<br>
6. Reordering columns logically in a better standard format:<br>
Source File: All 3 files<br>
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

Add a column
Source File: logistics_shipment_detail_3000.json
: domain as 'Logistics'

In [0]:
std_jsn=rawdf_jsn.withColumn("domain",lit("'Logistics'"))
display(std_jsn.take(20))

Column Uniformity: role - Convert to lowercase
Source File: logistics_source1 & logistics_source<br>2
vehicle_type - Convert values to UPPERCASE
Source Files: logistics_shipment_detail_3000.json <br>(and the merged master files) hub_location - Convert values to initcap case

In [0]:
def lowerad(col_value):
    conv_col_value=col_value.lower()
    return conv_col_value
print(lowerad("ASHOK"))

def upperad(col_value):
    conv_col_value=col_value.upper()
    return conv_col_value
print(upperad("ashok"))


In [0]:
from pyspark.sql.functions import udf,upper,initcap
udflower=udf(lowerad)
std_src=scrub_df4.withColumn("role",udflower(col("role")))
display(std_src)
std_jsn2=std_jsn.withColumn("vehicle_type",upper(col("vehicle_type")))
display(std_jsn2.take(20))
std_src2=std_src.withColumn("hub_location", initcap(col("hub_location")))
display(std_src2)

Format Standardization:<br>
Source Files: logistics_shipment_detail_3000.json<br> Convert shipment_ref to string<br>
Pad to 10 characters with leading zeros<br>
Convert dispatch_date to yyyy-MM-dd<br>
Ensure delivery_cost has 2 decimal precision

In [0]:
from pyspark.sql.functions import lpad,to_date,round
std_jsn3=std_jsn2.withColumn("shipment_id", lpad(col("shipment_id").cast("string"), 10, "0")).withColumn("shipment_date", to_date(col("shipment_date"), 'yy-MM-dd')).withColumn("shipment_cost", round(col("shipment_cost").cast("double"),2))
display(std_jsn3)

**Data Type Standardization**<br>
Standardizing column data types to fix schema drift and enable mathematical operations.<br>
Source File: logistics_source1 & logistics_source2
age: Cast String to Integer<br>
Source File: logistics_shipment_detail_3000.json
shipment_weight_kg: Cast to Double<br>
Source File: logistics_shipment_detail_3000.json
is_expedited: Cast to Boolean

In [0]:
std_src3=std_src2.withColumn("age",col("age").cast("int"))
display(std_src3.show(30,False)) #Datatype standardization
std_jsn4=std_jsn3.withColumn("shipment_weight_kg",col("shipment_weight_kg").cast("double")).withColumn("is_expedited",lit("False"))
display(std_jsn4)

Naming Standardization<br>
Source File: logistics_source1 & logistics_source2<br>
Rename: first_name to staff_first_name<br>
Rename: last_name to staff_last_name<br>
Rename: hub_location to origin_hub_city

In [0]:
std_src4=std_src3.withColumnsRenamed({"first_name":"staff_first_name","last_name":"staff_last_name","hub_location":"origin_hub_city"})
display(std_src4.show(30,False)) #Naming Standardization

Reordering columns logically in a better standard format:<br>
Source File: All 3 files<br>
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

In [0]:
print(std_src4.printSchema())
display(std_src4)
print("deduplicated count of records w.r to id col",std_src4.dropDuplicates(["shipment_id"]).count())
display(std_src4.groupBy("shipment_id").count().filter(col("count")>1))
print(std_jsn4.printSchema())

Deduplication:
1. Apply Record Level De-Duplication
2. Apply Column Level De-Duplication (Primary Key Enforcement)

Apply Record Level De-Duplication

In [0]:
print("total record count",std_src4.groupBy("shipment_id").count().count())
print("duplicated record counts w.r to id col",std_src4.groupBy("shipment_id").count().filter(col("count")>1).count())
print("deduplicated count of records w.r to id col",std_src4.dropDuplicates(["shipment_id"]).count())
dedupdf_src=std_src4.dropDuplicates()
display(dedupdf_src)
dedupdf_src=std_src4.dropDuplicates(["shipment_id"])
display(dedupdf_src.show(30,False))
print(dedupdf_src.count())
dedupdf_jsn=std_jsn4.dropDuplicates()
display(dedupdf_jsn)
display(dedupdf_jsn.groupBy("shipment_id").count().filter(col("count")>1))
dedupdf_jsn1=std_jsn4.dropDuplicates(["shipment_id"])
display(dedupdf_jsn1)
print("dedpulicated count of records w.r id col in jsn",dedupdf_jsn1.groupBy("shipment_id").count().count())


##2. Data Enrichment - Detailing of data
Makes your data rich and detailed <br>

###### Adding of Columns (Data Enrichment)
*Creating new derived attributes to enhance traceability and analytical capability.*

**1. Add Audit Timestamp (`load_dt`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
* **Action:** Add a column `load_dt` using the function `current_timestamp()`.

**2. Create Full Name (`full_name`)**
Source File: logistics_source1 and logistics_source2<br>
* **Scenario:** The reporting dashboard requires a single field for the driver's name instead of separate columns.
* **Action:** Create `full_name` by concatenating `first_name` and `last_name` with a space separator.
* **Result:** "Rajesh" + " " + "Kumar" -> **"Rajesh Kumar"**

**3. Define Route Segment (`route_segment`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
* **Action:** Combine `source_city` and `destination_city` with a hyphen.
* **Result:** "Chennai" + "-" + "Pune" -> **"Chennai-Pune"**

**4. Generate Vehicle Identifier (`vehicle_identifier`)**
Source File: logistics_shipment_detail_3000.json<br>
* **Scenario:** We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
* **Action:** Combine `vehicle_type` and `shipment_id` to create a composite key.
* **Result:** "Truck" + "_" + "500001" -> **"Truck_500001"**

Add Audit Timestamp (load_dt) 

In [0]:
from pyspark.sql.functions import current_timestamp,concat_ws,col
enrichdf_src1=dedupdf_src.withColumn("load_dt",lit(current_timestamp())).withColumn("full_name",concat_ws(" ",col("staff_first_name"),col("staff_last_name")))
display(enrichdf_src1)
enrichdf_jsn1=dedupdf_jsn1.withColumn("route_segment",concat_ws("-",col("source_city"),col("destination_city"))).withColumn("vehicle_identifier",concat_ws("_",col("vehicle_type"),col("shipment_id")))
display(enrichdf_jsn1)

###### Deriving of Columns (Time Intelligence)
*Extracting temporal features from dates to enable period-based analysis and reporting.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Derive Shipment Year (`shipment_year`)**
* **Scenario:** Management needs an annual performance report to compare growth year-over-year.
* **Action:** Extract the year component from `shipment_date`.
* **Result:** "2024-04-23" -> **2024**

**2. Derive Shipment Month (`shipment_month`)**
* **Scenario:** Analysts want to identify seasonal peaks (e.g., increased volume in December).
* **Action:** Extract the month component from `shipment_date`.
* **Result:** "2024-04-23" -> **4** (April)

**3. Flag Weekend Operations (`is_weekend`)**
* **Scenario:** The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
* **Action:** Flag as **'True'** if the `shipment_date` falls on a Saturday or Sunday.

In [0]:
from pyspark.sql.functions import year,to_date,col,month,date_format,concat,when
enrichdf_jsn2=enrichdf_jsn1.withColumn("shipment_year",
                                    year(to_date(col("shipment_date"), "yyyy-MM-dd"))).withColumn("shipment_month",
                                    concat(month(to_date(col("shipment_date"), "yyyy-MM-dd")),lit(" ("),
                                    date_format(to_date(col("shipment_date"), "yyyy-MM-dd"),"MMMM"),lit(")")))
                                                                                                                                                                           
display(enrichdf_jsn2)
enrichdf_jsn3=enrichdf_jsn2.withColumn("is_weekend",
        when(date_format(to_date(col("shipment_date"), "yyyy-MM-dd"), "E").isin("Sat","Sun"),lit("True")).otherwise(lit("False")))   
display(enrichdf_jsn3)                             

###### Enrichment/Business Logics (Calculated Fields)
*Deriving new metrics and financial indicators using mathematical and date-based operations.*<br>
Source File: logistics_shipment_detail_3000.json<br>

**1. Calculate Unit Cost (`cost_per_kg`)**
* **Scenario:** The Finance team wants to analyze the efficiency of shipments by determining the cost incurred per unit of weight.
* **Action:** Divide `shipment_cost` by `shipment_weight_kg`.
* **Logic:** `shipment_cost / shipment_weight_kg`

**2. Track Shipment Age (`days_since_shipment`)**
* **Scenario:** The Operations team needs to monitor how long it has been since a shipment was dispatched to identify potential delays.
* **Action:** Calculate the difference in days between the `current_date` and the `shipment_date`.
* **Logic:** `datediff(current_date(), shipment_date)`

**3. Compute Tax Liability (`tax_amount`)**
* **Scenario:** For invoicing and compliance, we must calculate the Goods and Services Tax (GST) applicable to each shipment.
* **Action:** Calculate 18% GST on the total `shipment_cost`.
* **Logic:** `shipment_cost * 0.18`

In [0]:
from pyspark.sql.functions import datediff
enrichdf_jsn4=enrichdf_jsn3.withColumn("cost_per_kg",when(col("shipment_weight_kg").cast("double")>0,round(col("shipment_cost").cast("double")/col("shipment_weight_kg").cast("double"),2)).otherwise(lit(0)))
display(enrichdf_jsn4.limit(20))
enrichdf_jsn5=enrichdf_jsn4.withColumn("days_since_shipment",datediff(current_timestamp(),to_date(col("shipment_date"),"yyyy-MM-dd")))
display(enrichdf_jsn5.limit(20))
enrichdf_jsn6=enrichdf_jsn5.withColumn("tax_amount",when(col("shipment_cost").cast("double")>0,round(col("shipment_cost").cast("double")*0.18,2)).otherwise(lit(0)))
display(enrichdf_jsn6.limit(20))

###### Remove/Eliminate (drop, select, selectExpr)
*Excluding unnecessary or redundant columns to optimize storage and privacy.*<br>
Source File: logistics_source1 and logistics_source2<br>

**1. Remove Redundant Name Columns**
* **Scenario:** Since we have already created the `full_name` column in the Enrichment step, the individual name columns are now redundant and clutter the dataset.
* **Action:** Drop the `first_name` and `last_name` columns.
* **Logic:** `df.drop("first_name", "last_name")`

In [0]:
#enrichdf_src2=enrichdf_src1.drop("staff_first_name","staff_last_name")
enrichdf_src2=enrichdf_src1.select("shipment_id","full_name","age","role","origin_hub_city","vehicle_type","data_source","load_dt")
display(enrichdf_src2.limit(20))

##### Splitting & Merging/Melting of Columns
*Reshaping columns to extract hidden values or combine fields for better analysis.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Splitting (Extraction)**
*Breaking one column into multiple to isolate key information.*
* **Split Order Code:**
  * **Action:** Split `order_id` ("ORD100000") into two new columns:
    * `order_prefix` ("ORD")
    * `order_sequence` ("100000")
* **Split Date:**
  * **Action:** Split `shipment_date` into three separate columns for partitioning:
    * `ship_year` (2024)
    * `ship_month` (4)
    * `ship_day` (23)

**2. Merging (Concatenation)**
*Combining multiple columns into a single unique identifier or description.*
* **Create Route ID:**
  * **Action:** Merge `source_city` ("Chennai") and `destination_city` ("Pune") to create a descriptive route key:
    * `route_lane` ("Chennai->Pune")

In [0]:
from pyspark.sql.functions import dayofmonth
enrichdf_jsn7=enrichdf_jsn6.withColumn("order_prefix",col("order_id").substr(1,3)).withColumn("order_sequence",col("order_id").substr(4,10)).withColumn("ship_day",dayofmonth(to_date(col("shipment_date"),"yyyy-MM-dd"))).withColumn("route_lane",concat_ws("->",col("source_city"),col("destination_city"))).drop("route_segment")
display(enrichdf_jsn7)
enricheddf_jsn=enrichdf_jsn7

## 3. Data Customization & Processing - Application of Tailored Business Specific Rules

### **UDF1: Complex Incentive Calculation**
**Scenario:** The Logistics Head wants to calculate a "Performance Bonus" for drivers based on tenure and role complexity.

**Action:** Create a Python function `calculate_bonus(role, age)` and register it as a Spark UDF.

**Logic:**
* **IF** `Role` == 'Driver' **AND** `Age` > 50:
  * `Bonus` = 15% of Salary (Reward for Seniority)
* **IF** `Role` == 'Driver' **AND** `Age` < 30:
  * `Bonus` = 5% of Salary (Encouragement for Juniors)
* **ELSE**:
  * `Bonus` = 0

**Result:** A new derived column `projected_bonus` is generated for every row in the dataset.

---

### **UDF2: PII Masking (Privacy Compliance)**
**Scenario:** For the analytics dashboard, we must hide the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.

**Business Rule:** Show the first 2 letters, mask the middle characters with `****`, and show the last letter.

**Action:** Create a UDF `mask_identity(name)`.

**Example:**
* **Input:** `"Rajesh"`
* **Output:** `"Ra****h"`
<br>
**Note: Convert the above udf logic to inbult function based transformation to ensure the performance is improved.**

In [0]:
from pyspark.sql.functions import udf,col
def bonus_cal(role,age):
    if role == "driver" and int(age) > 30:
        return "15% of salary"
    elif role == "driver" and int(age) < 30:
        return "5% of salary"
    else:
        return "No bonus"
bonus_udf = udf(bonus_cal)
custdf_src1=enrichdf_src2.withColumn("projected_bonus",udf(bonus_cal)(col("role"),col("age")))
display(custdf_src1)

In [0]:
def mask_identity(name):
    name = str(name).strip()
    n = len(name)
    return name[:2] + ("*"*(n-3)) + name[-1]
mask_id_udf = udf(mask_identity)
custdf_src2=custdf_src1.withColumn("full_name",mask_id_udf(col("full_name"))) 
display(custdf_src2)

## 4. Data Core Curation & Processing (Pre-Wrangling)
*Applying business logic to focus, filter, and summarize data before final analysis.*

**1. Select (Projection)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** The Driver App team only needs location data, not sensitive HR info.
* **Action:** Select only `first_name`, `role`, and `hub_location`.

**2. Filter (Selection)**<br>
Source File: json<br>
* **Scenario:** We need a report on active operational problems.
* **Action:** Filter rows where `shipment_status` is **'DELAYED'** or **'RETURNED'**.
* **Scenario:** Insurance audit for senior staff.
* **Action:** Filter rows where `age > 50`.

**3. Derive Flags & Columns (Business Logic)**<br>
Source File: json<br>
* **Scenario:** Identify high-value shipments for security tracking.
* **Action:** Create flag `is_high_value` = **True** if `shipment_cost > 50,000`.
* **Scenario:** Flag weekend operations for overtime calculation.
* **Action:** Create flag `is_weekend` = **True** if day is Saturday or Sunday.

**4. Format (Standardization)**<br>
Source File: json<br>
* **Scenario:** Finance requires readable currency formats.
* **Action:** Format `shipment_cost` to string like **"₹30,695.80"**.
* **Scenario:** Standardize city names for reporting.
* **Action:** Format `source_city` to Uppercase (e.g., "chennai" → **"CHENNAI"**).

**5. Group & Aggregate (Summarization)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** Regional staffing analysis.
* **Action:** Group by `hub_location` and **Count** the number of staff.
* **Scenario:** Fleet capacity analysis.
* **Action:** Group by `vehicle_type` and **Sum** the `shipment_weight_kg`.

**6. Sorting (Ordering)**<br>
Source File: json<br>
* **Scenario:** Prioritize the most expensive shipments.
* **Action:** Sort by `shipment_cost` in **Descending** order.
* **Scenario:** Organize daily dispatch schedule.
* **Action:** Sort by `shipment_date` (Ascending) then `priority_flag` (Descending).

**7. Limit (Top-N Analysis)**<br>
Source File: json<br>
* **Scenario:** Dashboard snapshot of critical delays.
* **Action:** Filter for 'DELAYED', Sort by Cost, and **Limit to top 10** rows.

In [0]:
select_df = custdf_src2.select("full_name", "role", "origin_hub_city")
display(select_df.limit(50))

In [0]:
curdf_jsn1 = enricheddf_jsn.filter(col("shipment_status").isin ("DELAYED","CANCELLED"))
display(curdf_jsn1.limit(20))
curdf_src1 = enrichdf_src2.filter(col("age") > 50)
display(curdf_src1.limit(20))

In [0]:
from pyspark.sql.functions import format_number,lit,col,upper
curdf_jsn2 = curdf_jsn1.withColumn("is_high_value",when(col("shipment_cost").cast("double")>40000,lit("True")).otherwise(lit("False")))
display(curdf_jsn2.limit(40))
formatdf_jsn = curdf_jsn2.withColumn("shipment_cost",concat(lit("₹"),format_number(col("shipment_cost").cast("double"),2)))
display(formatdf_jsn.limit(40))
formatdf_jsn2 = formatdf_jsn.withColumn("source_city",upper(col("source_city")))
display(formatdf_jsn2.limit(40))

5. Group & Aggregate (Summarization)
Source Files: logistics_source1 and logistics_source2

Scenario: Regional staffing analysis.
Action: Group by hub_location and Count the number of staff.
Scenario: Fleet capacity analysis.
Action: Group by vehicle_type and Sum the shipment_weight_kg.<br>
6. Sorting (Ordering)
Source File: json

Scenario: Prioritize the most expensive shipments.
Action: Sort by shipment_cost in Descending order.
Scenario: Organize daily dispatch schedule.
Action: Sort by shipment_date (Ascending) then priority_flag (Descending).<br>
7. Limit (Top-N Analysis)
Source File: json

Scenario: Dashboard snapshot of critical delays.
Action: Filter for 'DELAYED', Sort by Cost, and Limit to top 10 rows.

In [0]:
from pyspark.sql.functions import count,sum
curdf_src2=curdf_src1.groupBy("origin_hub_city").agg(count("*").alias("staff_count"))
display(curdf_src2.limit(40))
grpdf_jsn1=curdf_jsn2.groupBy("vehicle_type").agg(sum(col("shipment_weight_kg").cast("double")).alias("total_weight_kg"))
display(grpdf_jsn1.limit(40))

In [0]:
ship_costdf=curdf_jsn2.orderBy(col("shipment_cost").cast("double").desc())
display(ship_costdf.limit(40))

In [0]:
priority_df=curdf_jsn2.withColumn("priority_flag",when(col("shipment_cost").cast("double")>=50000, lit(3)).when(col("shipment_cost").cast("double")>=30000, lit(2)).otherwise(lit(1)))
display(priority_df.limit(20))
dispatch_df=priority_df.orderBy(to_date(col("shipment_date"),"yyyy-MM-dd").asc(),(col("priority_flag").desc()))
display(dispatch_df.limit(60))

In [0]:
topn_df=curdf_jsn2.filter(col("shipment_status") == "DELAYED").orderBy(col("shipment_cost").desc()).limit(10)
display(topn_df.limit(20))

## 5. Data Wrangling - Transformation & Analytics
*Combining, modeling, and analyzing data to answer complex business questions.*

### **1. Joins**
Source Files:<br>
Left Side (staff_df):<br> logistics_source1 & logistics_source2<br>
Right Side (shipments_df):<br> logistics_shipment_detail_3000.json<br>
#### **1.1 Frequently Used Simple Joins (Inner, Left)**
* **Inner Join (Performance Analysis):**
  * **Scenario:** We only want to analyze *completed work*. Connect Staff to the Shipments they handled.
  * **Action:** Join `staff_df` and `shipments_df` on `shipment_id`.
  * **Result:** Returns only rows where a staff member is assigned to a valid shipment.
* **Left Join (Idle Resource check):**
  * **Scenario:** Find out which staff members are currently *idle* (not assigned to any shipment).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right) on `shipment_id`. Filter where `shipments_df.shipment_id` is NULL.

#### **1.2 Infrequent Simple Joins (Self, Right, Full, Cartesian)**
* **Self Join (Peer Finding):**
  * **Scenario:** Find all pairs of employees working in the same `hub_location`.
  * **Action:** Join `staff_df` to itself on `hub_location`, filtering where `staff_id_A != staff_id_B`.
* **Right Join (Orphan Data Check):**
  * **Scenario:** Identify shipments in the system that have *no valid driver* assigned (Data Integrity Issue).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right). Focus on NULLs on the left side.
* **Full Outer Join (Reconciliation):**
  * **Scenario:** A complete audit to find *both* idle drivers AND unassigned shipments in one view.
  * **Action:** Perform a Full Outer Join on `shipment_id`.
* **Cartesian/Cross Join (Capacity Planning):**
  * **Scenario:** Generate a schedule of *every possible* driver assignment to *every* pending shipment to run an optimization algorithm.
  * **Action:** Cross Join `drivers_df` and `pending_shipments_df`.

#### **1.3 Advanced Joins (Semi and Anti)**
* **Left Semi Join (Existence Check):**
  * **Scenario:** "Show me the details of Drivers who have *at least one* shipment." (Standard filtering).
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_semi")`.
  * **Benefit:** Performance optimization; it stops scanning the right table once a match is found.
* **Left Anti Join (Negation Check):**
  * **Scenario:** "Show me the details of Drivers who have *never* touched a shipment."
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_anti")`.

### **2. Lookup**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Validation. Check if the `hub_location` in the staff file exists in the corporate `Master_City_List`.
* **Action:** Compare values against a reference list.

### **3. Lookup & Enrichment**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Geo-Tagging.
* **Action:** Lookup `hub_location` ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding `lat` and `long` columns for map plotting.

### **4. Schema Modeling (Denormalization)**<br>
Source Files: All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)<br>
* **Scenario:** Creating a "Gold Layer" Table for PowerBI/Tableau.
* **Action:** Flatten the Star Schema. Join `Staff`, `Shipments`, and `Vehicle_Master` into one wide table (`wide_shipment_history`) so analysts don't have to perform joins during reporting.

### **5. Windowing (Ranking & Trends)**<br>
Source Files:<br>
logistics_source2: Provides hub_location (Partition Key).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)<br>
* **Scenario:** "Who are the Top 3 Drivers by Cost in *each* Hub?"
* **Action:**
  1. Partition by `hub_location`.
  2. Order by `total_shipment_cost` Descending.
  3. Apply `dense_rank()` and `row_number()
  4. Filter where `rank or row_number <= 3`.

### **6. Analytical Functions (Lead/Lag)**<br>
Source File: <br>
logistics_shipment_detail_3000.json<br>
* **Scenario:** Idle Time Analysis.
* **Action:** For each driver, calculate the days elapsed since their *previous* shipment.

### **7. Set Operations**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Union:** Combining `Source1` (Legacy) and `Source2` (Modern) into one dataset (Already done in Active Munging).
* **Intersect:** Identifying Staff IDs that appear in *both* Source 1 and Source 2 (Duplicate/Migration Check).
* **Except (Difference):** Identifying Staff IDs present in Source 2 but *missing* from Source 1 (New Hires).

### **8. Grouping & Aggregations (Advanced)**<br>
Source Files:<br>
logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).<br>
* **Scenario:** The CFO wants a subtotal report at multiple levels:
  1. Total Cost by Hub.
  2. Total Cost by Hub AND Vehicle Type.
  3. Grand Total.
* **Action:** Use `cube("hub_location", "vehicle_type")` or `rollup()` to generate all these subtotals in a single query.

In [0]:
from pyspark.sql.functions import col
staff_df=scrub_df4
ship_df=std_jsn
innerdf=staff_df.join(ship_df,on="shipment_id",how="inner")
display(innerdf)
left_df=staff_df.alias("stf").join(ship_df.alias("ship"),on="shipment_id",how="left").where(col("ship.shipment_id").isNull())
display(left_df)

In [0]:
print("SELF")
self_df=staff_df.alias("st").join(staff_df.alias("sf"),on=(col("st.hub_location")==col("sf.hub_location")),how="inner").filter(col("st.shipment_id") != col("sf.shipment_id"))
self_df.select("st.shipment_id","sf.first_name","sf.hub_location").orderBy("hub_location").display()
print("Full Outer")
full_df=staff_df.alias("st").join(ship_df.alias("sf"),on=(col("st.shipment_id")==col("sf.shipment_id")),how="fullouter").display()
print("cartesian/cross")
cross_df=staff_df.join(ship_df).limit(20)
display(cross_df)

In [0]:
print("semi")
semi_df=staff_df.alias("st").join(ship_df.alias("sf"),on=col("st.shipment_id")==col("sf.shipment_id"),how="left_semi").display()

In [0]:
lf_antisemidf=staff_df.alias("st").join(ship_df.alias("sf"),on=col("st.shipment_id")==col("sf.shipment_id"),how="left_anti").display()

In [0]:
master_df=spark.read.csv("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/Master_City_List.csv", header=True, inferSchema=True)
display(master_df)
lkup_df=staff_df.alias("st").join(master_df.alias("ms"),on=col("st.hub_location")==col("ms.city_name"),how="left")
resultdf=lkup_df.select(staff_df["*"]).where(col("ms.city_name").isNotNull()).distinct()
print(resultdf.count())
enrch_lkdf=staff_df.alias("st").join(master_df.alias("ms"),on=col("st.hub_location")==col("ms.city_name"),how="left").select(col("st.*"),col("ms.latitude").alias("lat"),col("ms.longitude").alias("long"))
display(enrch_lkdf)

In [0]:
denorm_df=staff_df.alias("stf").join(ship_df.alias("ship"),on="shipment_id",how="left").display()

In [0]:
from pyspark.sql.functions import col, sum, dense_rank, rank, window, row_number, desc
from pyspark.sql.window import Window

#join_df = rawdf3.alias("src").join(ship_df.alias("sh"), on=(col("src.shipment_id") == col("sh.shipment_id")), how="inner")
join_df = rawdf3.join(ship_df,how="inner",on="shipment_id")
print(rawdf3.count())
display(join_df)
agg_df = join_df.filter(col("role") == "Driver") \
    .groupBy(col("hub_location"), col("first_name"), col("last_name")) \
    .agg(sum(col("shipment_cost").cast("double")).alias("total_ship_cost"))

display(agg_df)

rank_df = agg_df.withColumn(
    "dense_rank",
    dense_rank().over(Window.partitionBy("hub_location").orderBy(desc("total_ship_cost")))
).where("dense_rank<=3")

row_df = agg_df.withColumn(
    "seq_num",
    row_number().over(Window.partitionBy("hub_location").orderBy(col("total_ship_cost").cast("double").desc()))
)
display(rank_df)
display(row_df)

6. Analytical Functions (Lead/Lag)
Source File:
logistics_shipment_detail_3000.json

Scenario: Idle Time Analysis.
Action: For each driver, calculate the days elapsed since their previous shipment.

In [0]:
from pyspark.sql.functions import asc,window,col,lag,to_date,datediff
join_df2 = staff_df.join(ship_df,on="shipment_id",how="inner")
display(join_df2)
filter_df2 = join_df.filter(col("role") == "Driver")
display(filter_df2)
#df2 = filter_df2.withColumn(
#    "shipment_date",
#    to_date(col("shipment_date"),"yy-MM-dd")
#)
#display(df2)
wind_df = Window.partitionBy("role").orderBy(asc("shipment_date"))

lag_df = filter_df2.withColumn(
    "prev_shipment_date",
    lag(to_date(col("shipment_date"),"yy-MM-dd"),1).over(wind_df)
).withColumn(
    "idle_days",
    datediff(to_date(col("shipment_date"),"yy-MM-dd"),col("prev_shipment_date")
    )
)
display(lag_df)


7. Set Operations
Source Files: logistics_source1 and logistics_source2

Union: Combining Source1 (Legacy) and Source2 (Modern) into one dataset (Already done in Active Munging).<br>
Intersect: Identifying Staff IDs that appear in both Source 1 and Source 2 (Duplicate/Migration Check).<br>
Except (Difference): Identifying Staff IDs present in Source 2 but missing from Source 1 (New Hires).

In [0]:
inter_df = rawdf2.join(rawdf3,on="shipment_id",how="inner")
display(inter_df)
diff_df = rawdf3.join(rawdf2,on="shipment_id",how="semi")
display(diff_df)

8. Grouping & Aggregations (Advanced)
Source Files:
logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).<br>

Scenario: The CFO wants a subtotal report at multiple levels:
Total Cost by Hub.<br>
Total Cost by Hub AND Vehicle Type.<br>
Grand Total.
Action: Use cube("hub_location", "vehicle_type") or rollup() to generate all these subtotals in a single query.

In [0]:
#display(rawdf3)
#display(ship_df)
from pyspark.sql.functions import col,sum,when,round,lit
join_df3 = rawdf3.join(rawdf_jsn,on="shipment_id",how="inner").drop(rawdf_jsn["vehicle_type"])
display(join_df3)
clean_join_df = join_df3.filter(col("hub_location").isNotNull() | col("vehicle_type").isNotNull())
display(clean_join_df)
unknwn_df = clean_join_df.withColumn("vehicle_type",when(col("vehicle_type").isNull(),lit("Unknown")).otherwise(col("vehicle_type")))
display(unknwn_df)
agg_df4 = unknwn_df.rollup(col("hub_location"),col("vehicle_type")).agg(round(sum(col("shipment_cost").cast("double")),2).alias("total_ship_cost")).orderBy(col("hub_location").asc_nulls_last(),col("vehicle_type").asc_nulls_last())
display(agg_df4)
agg_df5 = unknwn_df.cube(col("hub_location"),col("vehicle_type")).agg(round(sum(col("shipment_cost").cast("double")),2).alias("total_ship_cost")).orderBy(col("hub_location").asc_nulls_last(),col("vehicle_type").asc_nulls_last())
display(agg_df5)
#orderBy(col("rd.hub_location").asc_nulls_last(),col("rd.vehicle_type").asc_nulls_last()))

##6. Data Persistance (LOAD)-> Data Publishing & Consumption<br>

Store the inner joined, lookup and enrichment, Schema Modeling, windowing, analytical functions, set operations, grouping and aggregation data into the delta tables.

In [0]:
join_df3.write.format("delta").mode("overwrite").save("/Volumes/logistic_catalog/default/logistic_volume/logistics_data/joined_df")
agg_df5.write.format("delta").mode("overwrite").saveAsTable("logistic_catalog.default.aggregated_tbl")

##7.Take the copy of the above notebook and try to write the equivalent SQL for which ever applicable.